In [ ]:
import numpy as np
import os
import pandas as pd

# to make this notebook's output stable across runs
np.random.seed(42)


In [ ]:
df_raw_h = pd.read_csv('brfss_2019_2025_harmonized.csv')
print("Data Raw shape:", df_raw_h.shape)
print("Data Raw columns:", df_raw_h.columns.tolist())
df_raw_h

In [ ]:
sentinel_codes = {
    "GENHLTH": {7: np.nan, 9: np.nan},
    "PHYSHLTH": {77: np.nan, 99: np.nan, 88: 0},
    "MENTHLTH": {77: np.nan, 99: np.nan, 88: 0},
    "_HLTHPL2": {9: np.nan},
    "PERSDOC3": {7: np.nan, 9: np.nan},
    "MEDCOST1": {7: np.nan, 9: np.nan},
    "CHECKUP1": {7: np.nan, 9: np.nan},
    "CVDSTRK3": {7: np.nan, 9: np.nan},
    "ADDEPEV3": {7: np.nan, 9: np.nan},
    "CHCKDNY2": {7: np.nan, 9: np.nan},
    "DIABETE4": {7: np.nan, 9: np.nan},
    "HAVARTH4": {7: np.nan, 9: np.nan},
    "MARITAL": {9: np.nan},
    "EDUCA": {9: np.nan},
    "RENTHOM1": {7: np.nan, 9: np.nan},
    "VETERAN3": {7: np.nan, 9: np.nan},
    "EMPLOY1": {9: np.nan},
    "INCOME3": {77: np.nan, 99: np.nan},
    "DIFFWALK": {7: np.nan, 9: np.nan},
    "SMOKE100": {7: np.nan, 9: np.nan},
    "ALCDAY4": {777: np.nan, 999: np.nan, 888: 0},
    "_MICHD": {9: np.nan},
    "_RACE": {9: np.nan},
    "_SEX": {9: np.nan},
    "_BMI5": {9999: np.nan},
    "_RFDRHV9": {9: np.nan},
    "_TOTINDA": {9: np.nan},
}

df_NaN = df_raw_h.replace(sentinel_codes)

In [ ]:
#Audit missingness (after converting each column's missing codes to NaN)

missing_summary = pd.DataFrame({
    'Missing_Count': df_NaN.isnull().sum(),
    'Missing_Percentage': (df_NaN.isnull().mean() * 100)
})

# Filter and sort columns with missing data greater than 0%
missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values(by='Missing_Percentage', ascending=False)
print(missing_summary)

In [ ]:
# List of low-relevance columns you want to eliminate entirely
columns_to_drop = ['_STATE']  # Add any other peripheral columns here

# FIX: start from df_NaN (missing codes already converted per column in cell 2).
# Previously this used df_raw_h, so the sentinel_codes mapping was never applied.
df_cleaned = df_NaN.drop(columns=columns_to_drop, errors='ignore')
print(f"Remaining columns: {df_cleaned.shape[1]}")

In [ ]:
# 1. Keep ALL rows here, including rows where DIABETE4 is missing.
#    Each model drops rows with a missing target itself. Keeping every row means the
#    18-29 files keep the same rows in the same order as before, so the shared
#    70/30 split file (split_18_29_70_30.csv) still lines up row by row.
df_clean = df_cleaned.copy()

# 2. For critical predictors with higher missingness (_BMI5, ALCDAY4, SMOKE100, INCOME3),
# create a missingness indicator flag (1 = missing: blank, Don't know or Refused)
for col in ['_BMI5', 'ALCDAY4', 'SMOKE100', 'INCOME3']:
    df_clean[f'{col}_missing'] = df_clean[col].isna().astype(int)

# 3. FIX: removed PHYSHLTH.fillna(88) and GENHLTH.fillna(median).
#    88 ("None") is already converted to 0 days by sentinel_codes, so filling with 88
#    would put back a fake "88 days". Missing values stay NaN here; tree models
#    impute inside their pipeline and df_modeling fills them below.

print(f"Original shape: {df_raw_h.shape}")
print(f"Cleaned shape preserving rows: {df_clean.shape}")

In [ ]:
# Scale down _BMI5 by 100
df_clean['_BMI5'] = df_clean['_BMI5'] / 100.0

# Quick check
print("BMI Sample Values (Scaled):")
print(df_clean['_BMI5'].head())

In [ ]:
#Unpack Complex Codes for ALCDAY4

def unpack_alcday(val):
    if pd.isna(val):
        return np.nan            # blank, 777 (Don't know) or 999 (Refused)
    elif val == 0 or val == 888:
        return 0                 # no drinks (888 is converted to 0 by sentinel_codes)
    elif 101 <= val <= 107:
        # Convert days per week to total days in past 30 days
        return (val - 100) * (30 / 7)
    elif 201 <= val <= 230:
        # Exact days in past month
        return val - 200
    else:
        return np.nan  # unexpected codes

# Apply the conversion
df_clean['ALCDAY4_cleaned'] = df_clean['ALCDAY4'].apply(unpack_alcday)

# Quick check of the new distribution
print(df_clean['ALCDAY4_cleaned'].describe())

In [ ]:
# Normalize two-category variables to 1/0 (1 = yes). Missing codes are already NaN.
yes_no_columns = ['MEDCOST1', 'CVDSTRK3', 'ADDEPEV3', 'CHCKDNY2', 'HAVARTH4',
                  'VETERAN3', 'DIFFWALK', '_MICHD', 'SMOKE100', '_HLTHPL2', '_TOTINDA']
binary_maps = {col: {1: 1, 2: 0} for col in yes_no_columns}   # 1 = Yes, 2 = No
binary_maps['_RFDRHV9'] = {1: 0, 2: 1}   # heavy drinker: raw 1 = No, 2 = Yes
binary_maps['_SEX']     = {1: 1, 2: 0}   # 1 = male, 0 = female
binary_maps['_URBSTAT'] = {1: 0, 2: 1}   # 1 = rural, 0 = urban
# _AGEG5YR is left as 1-14 because the age filter below uses those codes.

for col, mapping in binary_maps.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].map(mapping)   # anything not 1/2 becomes NaN

print("Binary column unique values after normalization:")
for col in binary_maps:
    if col in df_clean.columns:
        print(f"{col}: {sorted(df_clean[col].dropna().unique())}, NaN = {df_clean[col].isna().sum()}")

In [ ]:
#Apply Feature Scaling / Standardization (For PCA, Clustering, and Regressions)
from sklearn.preprocessing import StandardScaler

# Step 5: Scale continuous features for PCA, Clustering, and Regressions
scaling_cols = ['_BMI5', 'ALCDAY4_cleaned','EDUCA', 'PHYSHLTH', 'MENTHLTH']

# Fill missing values temporarily for scaling subset (using median)
scaler_df = df_clean[scaling_cols].copy()
for col in scaling_cols:
    scaler_df[col] = scaler_df[col].fillna(scaler_df[col].median())

# Initialize StandardScaler
scaler = StandardScaler()
scaled_array = scaler.fit_transform(scaler_df)

# Create a scaled dataframe for your distance/linear models
df_scaled = pd.DataFrame(scaled_array, columns=[f'{col}_scaled' for col in scaling_cols], index=df_clean.index)

print("Feature scaling complete. Shape of scaled dataframe:", df_scaled.shape)

In [ ]:
print("Df-clean shape is: ", df_clean.shape)
print("Df-scale shape is: ", df_scaled.shape)

In [ ]:
# Create df_modeling by combining your cleaned main dataset with your scaled features
# 1. Start with a copy of df_clean
df_modeling = df_clean.copy()

# 2. Drop the original unscaled continuous columns to avoid duplication
cols_to_replace = ['_BMI5', 'ALCDAY4_cleaned', 'EDUCA', 'PHYSHLTH', 'MENTHLTH']
df_modeling = df_modeling.drop(columns=cols_to_replace, errors='ignore')

# 3. Join your pristine df_scaled columns directly into df_modeling
df_modeling = pd.concat([df_modeling, df_scaled], axis=1)

# 4. Fill any remaining minor NaNs in categorical/binary columns with a placeholder (e.g., -1 or 0)
# so regression/classification models don't crash
df_modeling.fillna(-1, inplace=True)

print("df_modeling successfully created! Shape:", df_modeling.shape)

In [ ]:
# Save df_clean (optimized for XGBoost and Random Forest, retains NaNs and indicator flags)
#df_clean.to_csv('df_clean.csv', index=False)
#print("Saved df_clean.csv successfully!")

# Save df_scaled (the standardized continuous/ordinal features subset)
#df_scaled.to_csv('df_scaled.csv', index=False)
#print("Saved df_scaled.csv successfully!")

# Save df_modeling (fully complete, dense matrix with scaled features and filled NaNs for regressions/PCA/clustering)
#df_modeling.to_csv('df_modeling.csv', index=False)
#print("Saved df_modeling.csv successfully!")

In [ ]:
age_mask = df_clean['_AGEG5YR'].isin([1, 2])
selected_indices = df_clean.index[age_mask]

df_clean_18_29 = df_clean.loc[selected_indices].copy()
df_scaled_18_29 = df_scaled.loc[selected_indices].copy()
df_modeling_18_29 = df_modeling.loc[selected_indices].copy()

print("df_clean_18_29:", df_clean_18_29.shape)
print("df_scaled_18_29:", df_scaled_18_29.shape)
print("df_modeling_18_29:", df_modeling_18_29.shape)

# All three files must have the same rows in the same order as the shared split file
if os.path.exists("split_18_29_70_30.csv"):
    split_check = pd.read_csv("split_18_29_70_30.csv")
    assert len(split_check) == len(df_clean_18_29), "Row count no longer matches the split file"
    print("Row count matches split_18_29_70_30.csv")

In [ ]:
# Save df_clean (optimized for XGBoost and Random Forest, retains NaNs and indicator flags)
df_clean_18_29.to_csv('df_clean_18_29.csv', index=False)
print("Saved df_clean_18_29.csv successfully!")

# Save df_scaled (the standardized continuous/ordinal features subset)
df_scaled_18_29.to_csv('df_scaled_18_29.csv', index=False)
print("Saved df_scaled_18_29.csv successfully!")

# Save df_modeling (fully complete, dense matrix with scaled features and filled NaNs for regressions/PCA/clustering)
df_modeling_18_29.to_csv('df_modeling_18_29.csv', index=False)
print("Saved df_modeling_18_29.csv successfully!")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE



## Clustering - KMeans & KNN-based Analysis <a id='6'></a>

In [ ]:
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from IPython.display import display

# Load scaled features and matching labels
scaled_path = next(
    (path for path in ["df_scaled_18_29.csv", "df_scale_18_29.csv"] if os.path.exists(path)),
    None
)
if scaled_path is None:
    raise FileNotFoundError("Could not find df_scaled_18_29.csv or df_scale_18_29.csv.")

X_cluster = pd.read_csv(scaled_path)
df_labels = pd.read_csv("df_clean_18_29.csv")

if len(X_cluster) != len(df_labels):
    raise ValueError("Scaled features and cleaned data have different row counts.")

DATA = df_labels

# Same target and split for every model:
#   DIABETE4 1 = diabetes -> 1;  2 (pregnancy only), 3 (no), 4 (pre-diabetes) -> 0
#   rows with missing DIABETE4 (blank, 7 Don't know, 9 Refused) are dropped
#   train/test rows come from the shared 70/30 split file
split = pd.read_csv("split_18_29_70_30.csv")
assert len(split) == len(DATA), "Data rows do not match the split file"
target = pd.to_numeric(DATA["DIABETE4"], errors="coerce")
valid = target.isin([1, 2, 3, 4]).to_numpy()
is_train = (split["set"] == "train").to_numpy()
y_all = (target == 1).astype(int)

# KMeans clustering (unsupervised; diabetes is only used to describe the clusters)
km3 = KMeans(n_clusters=3, n_init=10, random_state=42)
clusters = km3.fit_predict(X_cluster)

cluster_summary = pd.DataFrame({
    "cluster": clusters[valid],
    "diabetes": y_all[valid].to_numpy()
}).groupby("cluster").agg(
    size=("diabetes", "size"),
    diabetes_rate=("diabetes", "mean")
)
print("KMeans cluster summary:")
display(cluster_summary)

# KNN classification: diabetes (1) vs no diabetes (0), shared 70/30 split
X_train_knn = X_cluster.loc[valid & is_train].reset_index(drop=True)
y_train_knn = y_all.loc[valid & is_train].reset_index(drop=True)
X_test_knn  = X_cluster.loc[valid & ~is_train].reset_index(drop=True)
y_test_knn  = y_all.loc[valid & ~is_train].reset_index(drop=True)

knn_model = make_pipeline(
    SimpleImputer(strategy="median"),
    KNeighborsClassifier(n_neighbors=5)
)
knn_model.fit(X_train_knn, y_train_knn)
y_pred_knn = knn_model.predict(X_test_knn)

print("KNN accuracy:", accuracy_score(y_test_knn, y_pred_knn))
print("KNN ROC-AUC:", roc_auc_score(y_test_knn, knn_model.predict_proba(X_test_knn)[:, 1]))
print(classification_report(
    y_test_knn, y_pred_knn, target_names=["No diabetes", "Diabetes"], zero_division=0
))

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# PCA projection used for plotting (fit on the scaled features)
pca_2d = PCA(n_components=2, random_state=42).fit(X_cluster)
centers_2d = pca_2d.transform(km3.cluster_centers_)
X_test_2d = pca_2d.transform(X_test_knn)
pc1_label = f"PC1 ({pca_2d.explained_variance_ratio_[0]:.0%} of variance)"
pc2_label = f"PC2 ({pca_2d.explained_variance_ratio_[1]:.0%} of variance)"

# Plot KMeans clusters and KNN actual/predicted labels in 2D PCA space
plot_idx = np.random.default_rng(42).choice(
    len(X_cluster), size=min(20_000, len(X_cluster)), replace=False
)
X_kmeans_2d = pca_2d.transform(X_cluster.iloc[plot_idx])

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

# KMeans clusters
for cluster_id in np.unique(clusters[plot_idx]):
    mask = clusters[plot_idx] == cluster_id
    axes[0].scatter(
        X_kmeans_2d[mask, 0], X_kmeans_2d[mask, 1],
        s=10, alpha=0.45, label=f"Cluster {cluster_id}"
    )

axes[0].scatter(
    centers_2d[:, 0], centers_2d[:, 1],
    marker="X", s=180, c="black", edgecolor="white", label="Centroids"
)
axes[0].set_title("KMeans clusters (PCA projection)")
axes[0].legend()

# KNN actual and predicted test labels
for ax, values, title in [
    (axes[1], y_test_knn.to_numpy(), "KNN: actual test labels"),
    (axes[2], y_pred_knn, "KNN: predicted test labels"),
]:
    for label, name, color in [
        (0, "No diabetes", "tab:blue"),
        (1, "Diabetes", "tab:orange"),
    ]:
        mask = values == label
        ax.scatter(
            X_test_2d[mask, 0], X_test_2d[mask, 1],
            s=12, alpha=0.5, color=color, label=name
        )
    ax.set_title(title)
    ax.legend()

for ax in axes:
    ax.set_xlabel(pc1_label)
    ax.set_ylabel(pc2_label)
    ax.grid(alpha=0.2)

plt.show()

In [ ]:
# Check which original features contribute most to PC1 and PC2
loadings = pd.DataFrame(
    pca_2d.components_,
    columns=X_cluster.columns,
    index=["PC1", "PC2"]
)
display(loadings)

In [ ]:
cluster_profiles = df_clean_18_29[scaling_cols].reset_index(drop=True).copy()
cluster_profiles["Cluster"] = clusters

print(cluster_profiles.groupby("Cluster")[scaling_cols].mean())

Diabetes is a chronic metabolic condition, whereas PHYSHLTH and MENTHLTH measure acute distress over just the past 30 days.Many individuals with long-standing, managed diabetes may report 0 days of poor physical or mental health in a given month (placing them on the left side of PC1), but they still have a high BMI or other underlying risk factors. Therefore, Cluster 0 successfully isolates a group that has a high concentration of diabetes not because they are sick every day, but because of other combined risk factors like body weight and age profiles.

## Decision Tree


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

df_tree = pd.read_csv("df_clean_18_29.csv")

DATA = df_tree

# Same target and split for every model:
#   DIABETE4 1 = diabetes -> 1;  2 (pregnancy only), 3 (no), 4 (pre-diabetes) -> 0
#   rows with missing DIABETE4 (blank, 7 Don't know, 9 Refused) are dropped
#   train/test rows come from the shared 70/30 split file
split = pd.read_csv("split_18_29_70_30.csv")
assert len(split) == len(DATA), "Data rows do not match the split file"
target = pd.to_numeric(DATA["DIABETE4"], errors="coerce")
valid = target.isin([1, 2, 3, 4]).to_numpy()
is_train = (split["set"] == "train").to_numpy()
y_all = (target == 1).astype(int)

X = df_tree.drop(columns=["DIABETE4"]).apply(pd.to_numeric, errors="coerce")

X_train, y_train = X.loc[valid & is_train], y_all.loc[valid & is_train]
X_test,  y_test  = X.loc[valid & ~is_train], y_all.loc[valid & ~is_train]

tree_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42)
)

tree_model.fit(X_train, y_train)
y_pred = tree_model.predict(X_test)

# Note: only ~1.5% of 18-29 year-olds have diabetes, so without class weights the tree
# predicts "No" for almost everyone. Accuracy is misleading here; look at ROC-AUC and recall.
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, tree_model.predict_proba(X_test)[:, 1]))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"], zero_division=0))

## Rule-Based

In [ ]:
from sklearn.tree import export_text

df_rules = pd.read_csv("df_clean_18_29.csv")

DATA = df_rules

# Same target and split for every model:
#   DIABETE4 1 = diabetes -> 1;  2 (pregnancy only), 3 (no), 4 (pre-diabetes) -> 0
#   rows with missing DIABETE4 (blank, 7 Don't know, 9 Refused) are dropped
#   train/test rows come from the shared 70/30 split file
split = pd.read_csv("split_18_29_70_30.csv")
assert len(split) == len(DATA), "Data rows do not match the split file"
target = pd.to_numeric(DATA["DIABETE4"], errors="coerce")
valid = target.isin([1, 2, 3, 4]).to_numpy()
is_train = (split["set"] == "train").to_numpy()
y_all = (target == 1).astype(int)

X_rules = df_rules.drop(columns=["DIABETE4"]).apply(pd.to_numeric, errors="coerce")

X_rules_train, y_rules_train = X_rules.loc[valid & is_train], y_all.loc[valid & is_train]
X_rules_test,  y_rules_test  = X_rules.loc[valid & ~is_train], y_all.loc[valid & ~is_train]

rule_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42
    )
)

rule_model.fit(X_rules_train, y_rules_train)
rule_predictions = rule_model.predict(X_rules_test)

print("Accuracy:", accuracy_score(y_rules_test, rule_predictions))
print("ROC-AUC:", roc_auc_score(y_rules_test, rule_model.predict_proba(X_rules_test)[:, 1]))
print("Confusion matrix:\n", confusion_matrix(y_rules_test, rule_predictions))
print(classification_report(
    y_rules_test,
    rule_predictions,
    labels=[0, 1],
    target_names=["No diabetes", "Diabetes"],
    zero_division=0
))

print("Decision rules:")
print(export_text(
    rule_model.named_steps["decisiontreeclassifier"],
    feature_names=list(X_rules.columns)
))

In [ ]:
from sklearn.tree import plot_tree

import matplotlib.pyplot as plt

tree = rule_model.named_steps["decisiontreeclassifier"]

plt.figure(figsize=(24, 12))
plot_tree(
    tree,
    feature_names=X_rules.columns,
    class_names=["No diabetes", "Diabetes"],
    filled=True,
    rounded=True,
    proportion=True,
    impurity=False,
    fontsize=9
)
plt.title("Rule-Based Decision Tree")
plt.tight_layout()
plt.show()